### Import packages needed

In [12]:
# Need to restart kernel if changes done in the .py importations!!

import sys,os
notebook_dir = os.getcwd()  
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
print(parent_dir)
sys.path.append(parent_dir)
# from implementation.Cloth import Cloth 
from implementation.Cloth_Ball import Cloth # need to call new code, which is in a new file!
from implementation.utils import createRectangularMesh, duplicate_node_pairs
import time
import numpy as np

/home/userlab/Desktop/clothilde-sim-ball-main/python_code


### Create or load an initial mesh

In [26]:
na = 30; nb = 18
X, T = createRectangularMesh(a = 0.9,b = 0.5, na = na, nb = nb, h = 0.05)
X[:,2] += 0.4; #adjust height

### Call the Cloth class

In [28]:
clothilde = Cloth(X, T)

### Call the Ball class inside Cloth class

In [29]:
# initialize ball inside Cloth class by caling method inside cloth
clothilde.ball = clothilde.addBall(position=[0,0.0,1], rad=0.04, mass=0.5, friction=0.2)

Check ball parameters are given correctly

In [30]:
print(f'Ball position: {clothilde.ball.position}')   # position
print(f'Ball speed: {clothilde.ball.velocity}')      # velocity
print(f'Ball radius: {clothilde.ball.rad}')    # radius
print(f'Ball friction: {clothilde.ball.mu_b}')     # friction
print(f'Ball position history: {clothilde.ball.history_pos}')  # full trajectory

Ball position: [0. 0. 1.]
Ball speed: [0. 0. 0.]
Ball radius: 0.04
Ball friction: 0.2
Ball position history: [array([0., 0., 1.])]


### Set simulator parameters:

In [31]:
# solver parameters
dt = 1/60 #frame rate
tol = 0.0095 # up to 0.75% of relative error in constraint satisfaction to stop iterations

#physical parameters
rho = 0.1 #cloth density
delta = 0.08 # aerodynamics parameter: between 0 and rho
kappa = 0.15*1e-4 # stifness or bending resistance
alpha = 0.2 #damping of oscillations
shr = 1.2*1e-4 #allowed shearing resistance
strh = 0.005*1e-4 #allowed stretching resistance
mu_f = 0.45 #friction with the floor
mu_s = 0.4 #friction with the cloth itself
thck = 0.9 #size of the balls
sub_steps = 12 #numer of intermidate steps between each dt

In [32]:
clothilde.setSimulatorParameters(dt=dt,tol=tol,sub_steps = sub_steps,
                                rho=rho,delta=delta,kappa=kappa,shr=shr,
                                str=strh,alpha=alpha,mu_f=mu_f,mu_s=mu_s,
                                thck=thck)

Plot meshes of both classes, need to call both, one for each!!

In [33]:
clothilde.preparePolyscope()
clothilde.ball.plotMesh()

### Let's start controling things

In [34]:
clothilde.restart()

In [35]:
inds_ctr = [0,na-1, (na*nb)-na, (na*nb)-1]
tf = int(2/dt)
u = X[inds_ctr]
for i in range(tf):
    clothilde.simulate(u = u, control = inds_ctr)

In [39]:
clothilde.makeMovie(1,True,2) # True/False to repeat simulation when finishes

In [38]:
print(f'Close Cloth nodes to Ball: {clothilde.ball.near_cloth}')

Close Cloth nodes to Ball: [224 225 253 254 255 256 283 284 285 286 314 315]


Save the video of the experiment

In [19]:
# Function to save frames
import polyscope as ps
import os
import numpy as np

def save_frames_ps(history_pos, Am, history_pos_ball, label, label_ball, folder="frames", step=4):
    os.makedirs(folder, exist_ok=True)
    n_frames = len(history_pos)
    print(f"Saving {n_frames//step} frames to {folder}/")

    for i, frame_idx in enumerate(range(0, n_frames, step)):
        phi_mat = history_pos[frame_idx]
        phi_all = Am @ phi_mat

        ps.get_surface_mesh(label).update_vertex_positions(phi_all)

        ball_pos = history_pos_ball[frame_idx]
        ps.get_point_cloud(label_ball).update_point_positions(ball_pos[np.newaxis, :])

        ps.screenshot(os.path.join(folder, f"frame_{i:04d}.png"), transparent_bg=False)


        if i % 50 == 0:
            print(f"Saved frame {i} / {n_frames//step}")

In [20]:
#  Function to save video from frames folder

import subprocess

def create_video_from_frames(frames_folder, output_video, framerate=20):
    """
    Create a video from a sequence of frames using ffmpeg.
    
    Args:
        frames_folder: Path to folder containing frame_XXXX.png files
        output_video: Output video filename (e.g., "experiment.mp4")
        framerate: Framerate for the video (default: 20)
    """
    try:
        cmd = [
            "ffmpeg",
            "-framerate", str(framerate),
            "-i", f"{frames_folder}/frame_%04d.png",
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-y",  # overwrite output file without asking
            output_video
        ]
        subprocess.run(cmd, check=True, capture_output=True)
        print(f"Video saved: {output_video}")
    except subprocess.CalledProcessError as e:
        print(f"Error creating video: {e.stderr.decode()}")
    except FileNotFoundError:
        print("ffmpeg not found. Install it using: brew install ffmpeg")

In [24]:
# save video

frames_folder = f"frames_Cloth_Ball_Temp"
os.makedirs(frames_folder, exist_ok=True)

# Fix camera once per experiment
ps.reset_camera_to_home_view()
ps.set_view_projection_mode("perspective")
ps.set_window_size(1854, 1011)

intrinsics = ps.CameraIntrinsics(fov_vertical_deg=45., fov_horizontal_deg=45.)

angle = np.radians(120)
radius = 2.0
height = 1.75

# Compute camera position
cam_x = radius * np.sin(angle)   # horizontal offset
cam_z = height                    # vertical offset (low, close to table)
cam_y = radius * np.cos(angle)    # forward/back offset

# Look at origin
look_dir = (-cam_x, -cam_y, -cam_z)
up_dir = (0, 0, 1) # with the mesh rotated, now the Y-axis is the original Z-axis, so we set up to be along global Z

extrinsics = ps.CameraExtrinsics(root=(cam_x, cam_y, cam_z), look_dir=look_dir, up_dir=up_dir)

ps.set_view_camera_parameters(ps.CameraParameters(intrinsics, extrinsics))

# edges configuration
mesh = ps.get_surface_mesh(clothilde.label)

mesh.set_color((0.2, 0.5, 0.8)) # RGB color of the mesh, blue for all meshes/cases (if not determined, each mesh for each new experiment will have different colors when polyscope registers them)
mesh.set_edge_color((0.0, 0.0, 0.0))  # RGB color of the edges, black for edges
mesh.set_edge_width(0.9)             # thickness of the edges

# ball configuration
ball = ps.get_point_cloud(clothilde.ball.label)
ball.set_color((0.8, 0.2, 0.2)) # RGB color of the ball = red

# Save frames
save_frames_ps(clothilde.history_pos, clothilde.Am, clothilde.ball.history_pos, clothilde.label, clothilde.ball.label, folder=frames_folder, step=8)

print(f"Finished saving frames")

# Create video from frames
create_video_from_frames(frames_folder, f"Cloth_Ball.mp4", framerate=20)

Saving 15 frames to frames_Cloth_Ball_Temp/
Saved frame 0 / 15
Finished saving frames
Video saved: Cloth_Ball.mp4


Check tree by printing the possible pairs and see if it matches with what seen in Polyscope by clicking on the nodes and see its position.


Check the substepping thing for ball.simulate(), as needed for better representations and synchronizing.

### Let's shake up things a little bit

In [11]:
t = np.linspace(0,2*np.pi,tf); freq = 2

In [12]:
inds_ctr = [0,na-1]
u = clothilde.positions[inds_ctr]; 
for i in range(tf):
    u[:,1] += 0.015*np.sin(freq*t[i])
    clothilde.simulate(u = u, control = inds_ctr)


In [29]:
clothilde.makeMovie(1,True,2)

### And now relax

In [35]:
inds_ctr = [0]
u = clothilde.positions[inds_ctr]; 
for i in range(tf):
    clothilde.simulate(u = u, control = inds_ctr)

In [36]:
clothilde.makeMovie(1,True,2)

In [37]:
print('Average iterations',clothilde.total_iters/(len(clothilde.history_pos)-1))


Average iterations 1.19375


### We can rerun everything with a different meshing and cloth!!